# Проверка витрины получателей рассылок

Ноутбук проверяет результаты расчёта `campaign_recipients`, сформированные DAG для трёх суточных интервалов 23–25 января 2026 года.

Проверки охватывают количество получателей по кампаниям, уникальность пары `(campaign_id, user_id)`, соответствие домена электронной почты и временную актуальность версии профиля SCD2 на момент отправки рассылки. Дополнительно приводится физический план JOIN.

В заключительной части используется отдельный набор `fixtures` для проверки граничных условий исторического JOIN и корректного сохранения пустого результата. Тестовые данные не изменяют основные результаты набора `demo`.


In [1]:
from jobs.common import create_spark_session, dataset_name, s3_uri
from pyspark.sql import functions as F

spark = create_spark_session("hw23-notebook-campaign-check")
dataset = dataset_name()
dates = ["2026-01-23", "2026-01-24", "2026-01-25"]


26/11/20 20:49:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Загрузка результатов расчёта

Из S3 загружаются три партиции витрины `campaign_recipients`, соответствующие датам расчёта 23, 24 и 25 января 2026 года. Далее выполняется просмотр схемы и примеров строк последней партиции.


In [2]:
recipients = {
    date: spark.read.parquet(
        s3_uri(f"results/{dataset}/campaign_recipients/calculation_date={date}")
    )
    for date in dates
}

df_recipients = recipients[dates[-1]]
df_recipients.printSchema()
df_recipients.orderBy("campaign_id", "user_id").show(30, truncate=False)


26/11/20 20:49:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- campaign_id: long (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- send_at: timestamp (nullable = true)
 |-- email_domain: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)
 |-- user_valid_from: timestamp (nullable = true)
 |-- user_valid_to: timestamp (nullable = true)
 |-- calculation_date: date (nullable = true)
 |-- _dag_id: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _task_id: string (nullable = true)
 |-- _data_interval_start: timestamp (nullable = true)
 |-- _data_interval_end: timestamp (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)

+-----------+-------------+-------------------+---------------+-------+---------------+--------------------------------------+--------------+-------------------+-------------------+----------------+------------------------+-----------------------

## Количество получателей по кампаниям

Для каждой даты расчёта определяется общее количество строк витрины и количество получателей в разрезе `campaign_id` и `campaign_name`. Одна строка витрины соответствует одному получателю одной рассылки.


In [3]:
for date, df in recipients.items():
    print(f"\nДата расчёта: {date}; всего строк: {df.count()}")
    df.groupBy("campaign_id", "campaign_name").count().orderBy("campaign_id").show(100, truncate=False)



Дата расчёта: 2026-01-23; всего строк: 53
+-----------+-------------+-----+
|campaign_id|campaign_name|count|
+-----------+-------------+-----+
|11         |Campaign_0011|10   |
|16         |Campaign_0016|23   |
|23         |Campaign_0023|20   |
+-----------+-------------+-----+


Дата расчёта: 2026-01-24; всего строк: 25
+-----------+-------------+-----+
|campaign_id|campaign_name|count|
+-----------+-------------+-----+
|21         |Campaign_0021|25   |
+-----------+-------------+-----+


Дата расчёта: 2026-01-25; всего строк: 22
+-----------+-------------+-----+
|campaign_id|campaign_name|count|
+-----------+-------------+-----+
|19         |Campaign_0019|22   |
+-----------+-------------+-----+



## Проверка уникальности и условий исторического JOIN

Для каждой даты проверяются три инварианта:

1. Отсутствуют повторяющиеся пары `(campaign_id, user_id)`.
2. Домен `user_email` совпадает с `email_domain` без учёта регистра.
3. Момент `send_at` принадлежит интервалу действия присоединённой версии профиля: `user_valid_from <= send_at < user_valid_to`; для текущей версии верхняя граница отсутствует.

Каждая проверка завершается утверждением, поэтому нарушение любого условия приводит к ошибке выполнения ячейки.


In [4]:
for date, df in recipients.items():
    duplicate_pairs = (
        df.groupBy("campaign_id", "user_id").count()
        .filter(F.col("count") > 1).count()
    )
    wrong_domain = df.filter(
        F.lower(F.trim(F.split("user_email", "@").getItem(1))) !=
        F.lower(F.trim(F.col("email_domain")))
    ).count()
    outside_validity = df.filter(
        (F.col("user_valid_from") > F.col("send_at")) |
        (F.col("user_valid_to").isNotNull() & (F.col("send_at") >= F.col("user_valid_to")))
    ).count()
    print(date, {
        "duplicate_campaign_user_pairs": duplicate_pairs,
        "wrong_email_domain": wrong_domain,
        "outside_scd2_validity": outside_validity,
    })
    assert duplicate_pairs == 0
    assert wrong_domain == 0
    assert outside_validity == 0


2026-01-23 {'duplicate_campaign_user_pairs': 0, 'wrong_email_domain': 0, 'outside_scd2_validity': 0}
2026-01-24 {'duplicate_campaign_user_pairs': 0, 'wrong_email_domain': 0, 'outside_scd2_validity': 0}
2026-01-25 {'duplicate_campaign_user_pairs': 0, 'wrong_email_domain': 0, 'outside_scd2_validity': 0}


## Физический план выполнения JOIN

Для одной даты повторно формируется JOIN между снимком SCD2 и кампаниями с использованием `broadcast` для небольшой таблицы рассылок. Метод `explain(mode="formatted")` выводит физический план Spark.

План используется для подтверждения выбранной стратегии соединения. Ожидаемый для данной реализации оператор — `BroadcastHashJoin`; фактический план определяется Spark во время выполнения.


In [5]:
date = dates[0]
scd2 = spark.read.parquet(
    s3_uri(f"results/{dataset}/user_scd2/snapshot_date={date}")
).alias("u")
campaigns = (
    spark.read.option("header", "true").csv(s3_uri(f"raw/{dataset}/campaigns.csv"))
    .withColumn("campaign_id", F.col("campaign_id").cast("long"))
    .withColumn("send_at", F.to_timestamp("send_at", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .filter((F.col("send_at") >= F.to_timestamp(F.lit(date + " 00:00:00"))) &
            (F.col("send_at") < F.to_timestamp(F.lit(date + " 00:00:00")) + F.expr("INTERVAL 1 DAY")))
    .alias("c")
)
condition = (
    (F.lower(F.split(F.col("u.user_email"), "@").getItem(1)) == F.lower(F.col("c.email_domain"))) &
    (F.col("u.valid_from") <= F.col("c.send_at")) &
    (F.col("u.valid_to").isNull() | (F.col("c.send_at") < F.col("u.valid_to")))
)
plan_df = scd2.join(F.broadcast(campaigns), condition, "inner")
plan_df.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (8)
+- BroadcastHashJoin Inner BuildRight (7)
   :- Filter (2)
   :  +- Scan parquet  (1)
   +- BroadcastExchange (6)
      +- Project (5)
         +- Filter (4)
            +- Scan csv  (3)


(1) Scan parquet 
Output [14]: [user_id#579L, user_fio#580, user_email#581, user_login#582, valid_from#583, valid_to#584, is_current#585, snapshot_date#586, _dag_id#587, _run_id#588, _task_id#589, _data_interval_start#590, _data_interval_end#591, _loaded_at#592]
Batched: true
Location: InMemoryFileIndex [s3a://hw23/results/demo/user_scd2/snapshot_date=2026-01-23]
PushedFilters: [IsNotNull(user_email), IsNotNull(valid_from)]
ReadSchema: struct<user_id:bigint,user_fio:string,user_email:string,user_login:string,valid_from:timestamp,valid_to:timestamp,is_current:boolean,snapshot_date:date,_dag_id:string,_run_id:string,_task_id:string,_data_interval_start:timestamp,_data_interval_end:timestamp,_loaded_at:timestamp>

(2) Filter
Input [14]: [user_id#579L, user_fio#5

## Граничные случаи исторического JOIN на наборе fixtures

Для независимой проверки используется учебный набор `fixtures`, содержащий 14 событий профилей и 16 рассылок. По событиям строится тестовая история SCD2: повтор события с неизменившимися атрибутами не создаёт новую версию, а изменение ФИО, email или login начинает новую версию.

История строится до `2026-01-28 00:00:00 UTC`, чтобы включить все события и контрольные рассылки. Основные партиции `demo` не изменяются.

Контрольные рассылки охватывают смену домена, точное совпадение `send_at` с `valid_from`, момент непосредственно перед границей версии, момент до первого события пользователя, смену состояния на границе суток, повтор состояния и сравнение домена без учёта регистра.


In [6]:
from pyspark.sql.window import Window

FIXTURE_SNAPSHOT_END = "2026-01-28 00:00:00"
fixtures_events_path = s3_uri("raw/fixtures/events/*.csv")
fixtures_campaigns_path = s3_uri("raw/fixtures/campaigns.csv")

fixture_raw_events = (
    spark.read.option("header", "true").option("inferSchema", "false")
    .csv(fixtures_events_path)
)
fixture_events = fixture_raw_events.select(
    F.to_timestamp(F.trim("event_time"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("event_time"),
    F.col("user_id").cast("long").alias("user_id"),
    F.col("user_fio").cast("string").alias("user_fio"),
    F.col("user_email").cast("string").alias("user_email"),
    F.col("user_login").cast("string").alias("user_login"),
).filter(F.col("event_time") < F.lit(FIXTURE_SNAPSHOT_END).cast("timestamp"))

# SCD2 повторяет алгоритм расчётной задачи: сортировка по времени,
# сравнение полного состояния и создание версии только при изменении атрибутов.
w_events = Window.partitionBy("user_id").orderBy("event_time")
fixture_changed = (
    fixture_events
    .withColumn("_profile", F.struct("user_fio", "user_email", "user_login"))
    .withColumn("_previous_profile", F.lag("_profile").over(w_events))
    .filter(
        F.col("_previous_profile").isNull()
        | (~F.col("_profile").eqNullSafe(F.col("_previous_profile")))
    )
)
w_versions = Window.partitionBy("user_id").orderBy("valid_from")
fixture_scd2 = fixture_changed.select(
    "user_id", "user_fio", "user_email", "user_login",
    F.col("event_time").alias("valid_from"),
).withColumn("valid_to", F.lead("valid_from").over(w_versions))
fixture_scd2 = fixture_scd2.withColumn("is_current", F.col("valid_to").isNull())

print("Событий до конца тестового снимка:", fixture_events.count())
print("Версий SCD2:", fixture_scd2.count())
print("Пользователей:", fixture_scd2.select("user_id").distinct().count())
fixture_scd2.orderBy("user_id", "valid_from").show(30, truncate=False)

# В fixtures 14 событий, из которых два повторяют предыдущее состояние;
# поэтому ожидается 12 версий для пяти пользователей.
assert fixture_events.count() == 14, "Проверьте, что загружен исходный набор fixtures (14 событий)"
assert fixture_scd2.count() == 12, "Ожидалось 12 версий SCD2 после удаления повторов состояния"
assert fixture_scd2.select("user_id").distinct().count() == 5

# Проверка уникальности ключа, ровно одной текущей версии и отсутствия разрывов/пересечений.
assert fixture_scd2.groupBy("user_id", "valid_from").count().filter("count > 1").count() == 0
current_per_user = fixture_scd2.groupBy("user_id").agg(
    F.sum(F.col("is_current").cast("int")).alias("current_count")
)
assert current_per_user.filter(F.col("current_count") != 1).count() == 0
w_check = Window.partitionBy("user_id").orderBy("valid_from")
fixture_intervals = fixture_scd2.withColumn("_next_valid_from", F.lead("valid_from").over(w_check))
assert fixture_intervals.filter(
    F.col("_next_valid_from").isNotNull() & (F.col("valid_to") != F.col("_next_valid_from"))
).count() == 0
print("Проверки SCD2 на fixtures: OK")

Событий до конца тестового снимка: 14
Версий SCD2: 12
Пользователей: 5
+-------+-------------+--------------------+------------+-------------------+-------------------+----------+
|user_id|user_fio     |user_email          |user_login  |valid_from         |valid_to           |is_current|
+-------+-------------+--------------------+------------+-------------------+-------------------+----------+
|1001   |Alice One    |alice@example.com   |alice       |2026-01-22 08:00:00|2026-01-23 12:00:00|false     |
|1001   |Alice One    |alice@mail.example  |alice       |2026-01-23 12:00:00|2026-01-24 09:00:00|false     |
|1001   |Alicia One   |alice@mail.example  |alice       |2026-01-24 09:00:00|2026-01-25 12:00:00|false     |
|1001   |Alicia One   |alice@mail.example  |alice_new   |2026-01-25 12:00:00|2026-01-25 18:00:00|false     |
|1001   |Alicia One   |alice@mail.example  |alice       |2026-01-25 18:00:00|2026-01-26 00:00:00|false     |
|1001   |Alicia Future|alice@future.example|alice_future|

## Результаты проверки основной витрины

Для каждой даты отображаются количество строк и распределение получателей по кампаниям, а также результаты проверок уникальности, домена и временной актуальности профиля.

Физический план JOIN выводится отдельно и позволяет определить фактически выбранный Spark оператор соединения.


## Контрольные результаты JOIN на fixtures

Для каждой контрольной кампании сравнивается фактический список `user_id` с ожидаемым списком, определённым по исходным событиям и времени отправки. Проверяются как положительные, так и отрицательные случаи, включая отсутствие профиля или совпадающего домена.

Сравнение выполняется по полным спискам идентификаторов, а не только по количеству строк.


In [7]:
fixture_campaigns_raw = (
    spark.read.option("header", "true").option("inferSchema", "false")
    .csv(fixtures_campaigns_path)
)
fixture_campaigns = fixture_campaigns_raw.select(
    F.col("campaign_id").cast("long").alias("campaign_id"),
    F.col("campaign_name").cast("string").alias("campaign_name"),
    F.to_timestamp(F.trim("send_at"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("send_at"),
    F.col("email_domain").cast("string").alias("email_domain"),
)

u = fixture_scd2.alias("u")
c = fixture_campaigns.alias("c")
fixture_join_condition = (
    (F.lower(F.trim(F.split(F.col("u.user_email"), "@").getItem(1)))
     == F.lower(F.trim(F.col("c.email_domain"))))
    & (F.col("u.valid_from") <= F.col("c.send_at"))
    & (F.col("u.valid_to").isNull() | (F.col("c.send_at") < F.col("u.valid_to")))
)
fixture_join = u.join(c, fixture_join_condition, "inner").select(
    F.col("c.campaign_id").alias("campaign_id"),
    F.col("c.campaign_name").alias("campaign_name"),
    F.col("c.send_at").alias("send_at"),
    F.col("c.email_domain").alias("email_domain"),
    F.col("u.user_id").alias("user_id"),
    F.col("u.user_fio").alias("user_fio"),
    F.col("u.user_email").alias("user_email"),
    F.col("u.user_login").alias("user_login"),
    F.col("u.valid_from").alias("user_valid_from"),
    F.col("u.valid_to").alias("user_valid_to"),
)

actual_by_campaign = {
    row["campaign_id"]: sorted(row["user_ids"] or [])
    for row in (
        fixture_join.groupBy("campaign_id")
        .agg(F.collect_list("user_id").alias("user_ids"))
        .collect()
    )
}
# Явно добавляем кампании с нулём получателей, которых нет в inner join.
fixture_campaign_ids = [row["campaign_id"] for row in fixture_campaigns.select("campaign_id").collect()]
for campaign_id in fixture_campaign_ids:
    actual_by_campaign.setdefault(campaign_id, [])

expected_by_campaign = {
    101: [1001],       # До смены домена действует example.com.
    102: [1002],       # В 12:00 старая версия 1001 уже закрыта; 1002 появляется с example.com.
    103: [1001],       # Ровно в valid_from 12:00 уже действует mail.example.
    104: [],            # За секунду до смены домена mail.example ещё не действует.
    105: [1001, 1002],  # В полночь 1002 уже перешёл на mail.example.
    106: [1003],        # Точное время первого события 1003; Inbox.Example сравнивается без регистра.
    107: [],            # За секунду до первого события 1003 профиль ещё не существует.
    108: [1002],        # Ровно в полночь 1002 перешёл на post.example.
    109: [1001],        # Повтор состояния не создаёт новую версию.
    110: [1004],        # Ровно в valid_from первого события 1004.
    111: [1004, 1005], # 1004 продолжает действовать; 1005 появляется ровно в valid_from.
    112: [1001],        # После первого события 1001, до основного окна расчёта.
    113: [],            # Нет пользователей с absent.example.
    114: [],            # Пустой результат 27 января.
    115: [1001],        # Сравнение домена без учёта регистра.
    116: [1001],        # Состояние 1001 после изменения login.
}

for campaign_id, expected_users in expected_by_campaign.items():
    actual_users = actual_by_campaign.get(campaign_id, [])
    assert actual_users == expected_users, (
        f"campaign_id={campaign_id}: ожидались {expected_users}, получены {actual_users}"
    )

print("Контрольные примеры JOIN на fixtures:")
for campaign_id in sorted(expected_by_campaign):
    print(f"campaign_id={campaign_id}: user_id={actual_by_campaign[campaign_id]}")
print("Все ожидаемые наборы получателей совпали.")

Контрольные примеры JOIN на fixtures:
campaign_id=101: user_id=[1001]
campaign_id=102: user_id=[1002]
campaign_id=103: user_id=[1001]
campaign_id=104: user_id=[]
campaign_id=105: user_id=[1001, 1002]
campaign_id=106: user_id=[1003]
campaign_id=107: user_id=[]
campaign_id=108: user_id=[1002]
campaign_id=109: user_id=[1001]
campaign_id=110: user_id=[1004]
campaign_id=111: user_id=[1004, 1005]
campaign_id=112: user_id=[1001]
campaign_id=113: user_id=[]
campaign_id=114: user_id=[]
campaign_id=115: user_id=[1001]
campaign_id=116: user_id=[1001]
Все ожидаемые наборы получателей совпали.


## Пустой результат и схема Parquet

Для кампании `114` ожидается отсутствие получателей. Проверяется, что пустой результат сохраняет полную схему витрины, включая дату расчёта и шесть обязательных метаполей.

Пустой DataFrame записывается во временный каталог в формате Parquet, затем повторно читается. Проверяются нулевое количество строк и сохранность всех столбцов схемы.


In [8]:
from datetime import datetime
import os
import shutil

fixture_result_schema_df = fixture_join.select(
    "campaign_id", "campaign_name", "send_at", "email_domain", "user_id",
    "user_fio", "user_email", "user_login", "user_valid_from", "user_valid_to",
).withColumn("calculation_date", F.lit("2026-01-27").cast("date"))

# Для проверки схемы добавляем шесть метаполей с тестовыми значениями.
fixture_result_schema_df = (
    fixture_result_schema_df
    .withColumn("_dag_id", F.lit("fixture_validation").cast("string"))
    .withColumn("_run_id", F.lit("fixture_validation_run").cast("string"))
    .withColumn("_task_id", F.lit("fixture_join_check").cast("string"))
    .withColumn("_data_interval_start", F.lit("2026-01-27 00:00:00").cast("timestamp"))
    .withColumn("_data_interval_end", F.lit("2026-01-28 00:00:00").cast("timestamp"))
    .withColumn("_loaded_at", F.current_timestamp().cast("timestamp"))
)
empty_fixture_result = fixture_result_schema_df.filter(F.col("campaign_id") == 114)
assert empty_fixture_result.count() == 0

expected_result_columns = [
    "campaign_id", "campaign_name", "send_at", "email_domain", "user_id",
    "user_fio", "user_email", "user_login", "user_valid_from", "user_valid_to",
    "calculation_date", "_dag_id", "_run_id", "_task_id",
    "_data_interval_start", "_data_interval_end", "_loaded_at",
]
assert empty_fixture_result.columns == expected_result_columns

temp_empty_path = "/tmp/hw23_fixture_empty_campaign_recipients_parquet"
if os.path.exists(temp_empty_path):
    shutil.rmtree(temp_empty_path)
empty_fixture_result.write.mode("overwrite").parquet(temp_empty_path)
empty_roundtrip = spark.read.parquet(temp_empty_path)
assert empty_roundtrip.count() == 0
assert empty_roundtrip.columns == expected_result_columns
print("Пустой результат записан и прочитан как Parquet.")
print("Количество строк:", empty_roundtrip.count())
print("Количество столбцов:", len(empty_roundtrip.columns))
empty_roundtrip.printSchema()

Пустой результат записан и прочитан как Parquet.
Количество строк: 0
Количество столбцов: 17
root
 |-- campaign_id: long (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- send_at: timestamp (nullable = true)
 |-- email_domain: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)
 |-- user_valid_from: timestamp (nullable = true)
 |-- user_valid_to: timestamp (nullable = true)
 |-- calculation_date: date (nullable = true)
 |-- _dag_id: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _task_id: string (nullable = true)
 |-- _data_interval_start: timestamp (nullable = true)
 |-- _data_interval_end: timestamp (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)

